# 15 Similar is not relevant: what a sentence transformer ranks and what a cross-encoder decides

A sentence transformer (a **bi-encoder**) turns the question and each document into vectors separately, so every document is embedded once, at index time, and a question costs one pass plus a dot product with each stored vector. It ranks by closeness in meaning. A **cross-encoder** reads the question and one document together and returns a relevance score; nothing can be precomputed, so it costs one pass per pair and runs on a short list. **BM25** counts shared words, weighted so that rare words count more, and knows nothing about meaning. Hybrid retrieval merges the BM25 and sentence-transformer rankings by **reciprocal rank fusion**: each document scores the sum of 1/(60 + rank) over the lists, so the two scores need no common scale.

This notebook asks three questions on the Meridian policy and destination documents: which ranker puts the right document first; what happens when a superseded document says something different from the current one; and whether either score can gate "the documents do not answer this" without a source of truth other than the score itself.

**What the numbers cover.** 31 whole documents (not chunks), seven of them superseded or agent-only; the 20 questions of the RAG workshop's shipped evaluation set, each with gold document ids, three of them stale traps; and six questions written for this notebook that no document answers, three of which share a topic with a document (Oslo, Tromso). The models are bge-small (the bi-encoder) and bge-reranker-base (the cross-encoder). With 20 questions the intervals are wide and a difference between two rankers is an illustration. The corpus comes from the author's RAG workshop (the 36-question set used there is not part of this repo), so the figures here are not that workshop's figures.

In [1]:
import json, math, re, time
from collections import Counter
from pathlib import Path
import numpy as np, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics import roc_auc_score
import meridian as m

torch.set_num_threads(3)
RAG = Path("data/rag/meridian")

def load_docs():
    docs = []
    for p in sorted(RAG.glob("*/*.md")):
        head, body = p.read_text().split("---\n", 2)[1:]
        meta = dict(l.split(": ", 1) for l in head.strip().splitlines() if ": " in l)
        docs.append({"id": meta["doc_id"], "title": meta["title"], "current": meta["is_current"] == "true",
                     "audience": meta["audience"], "text": meta["title"] + ". " + " ".join(body.split())})
    return docs

docs = load_docs()
ids = [d["id"] for d in docs]
bad = {d["id"] for d in docs if not d["current"] or d["audience"] != "customer"}   # superseded or agent-only: must not reach a customer answer
qs = json.load(open(RAG / "eval_set.json"))
UNANSWERABLE = ["How do I reset the password on my Meridian account?",
                "What time does the Oslo office open on Saturdays?",
                "Who won the football match last night?",
                "What is the weather forecast for Oslo next week?",
                "Is the northern lights tour in Tromso wheelchair accessible?",
                "How much does a hotel room in Paris cost tonight?"]
print(f"{len(docs)} documents ({len(bad)} superseded or agent-only: {sorted(bad)})")
print(f"{len(qs)} answerable questions with gold documents, {len(UNANSWERABLE)} questions the corpus cannot answer (written for this notebook)")

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


31 documents (7 superseded or agent-only: ['baggage_2019_outdated', 'health_bali_2023_archived', 'health_covid_archived', 'sop_complaints', 'sop_escalation', 'sop_refunds', 'visa_schengen_2023_archived'])
20 answerable questions with gold documents, 6 questions the corpus cannot answer (written for this notebook)


**Goal.** Load the documents and questions, and mark which documents a customer answer must never use.

In [2]:
tokenize = lambda t: re.findall(r"[a-z0-9]+", t.lower())

class BM25:
    """Okapi BM25 over whole documents: counts shared words, weighting rare ones, and knows nothing about meaning."""
    def __init__(self, texts, k1=1.5, b=0.75):
        self.tf = [Counter(tokenize(t)) for t in texts]; self.len = np.array([sum(c.values()) for c in self.tf])
        df = Counter(w for c in self.tf for w in c); n = len(texts)
        self.idf = {w: math.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}
        self.k1, self.b, self.avg = k1, b, self.len.mean()
    def scores(self, q):
        s = np.zeros(len(self.tf))
        for w in tokenize(q):
            if w not in self.idf: continue
            for i, c in enumerate(self.tf):
                if w in c: s[i] += self.idf[w] * c[w] * (self.k1 + 1) / (c[w] + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s

bm25 = BM25([d["text"] for d in docs])
bi = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu", local_files_only=True)
ce = CrossEncoder("BAAI/bge-reranker-base", device="cpu", local_files_only=True)
D = bi.encode([d["text"] for d in docs], normalize_embeddings=True)          # every document embedded once, at index time
PREFIX = "Represent this sentence for searching relevant passages: "
rank = lambda s: [ids[i] for i in np.argsort(-s)]

def rrf(*lists, k=60):
    s = Counter()
    for l in lists:
        for r, d in enumerate(l): s[d] += 1 / (k + r + 1)
    return [d for d, _ in s.most_common()]

def run(q, rerank_k=10):
    qv = bi.encode(PREFIX + q, normalize_embeddings=True); cos = D @ qv
    r_bm, r_de = rank(bm25.scores(q)), rank(cos)
    fused = rrf(r_bm, r_de)
    pool = fused[:rerank_k]
    rel = ce.predict([(q, docs[ids.index(d)]["text"]) for d in pool], show_progress_bar=False)   # one pass per (query, document) pair
    r_ce = [pool[i] for i in np.argsort(-rel)]
    return {"bm25": r_bm, "dense": r_de, "fused": fused, "rerank": r_ce, "cos_top": float(cos.max()), "rel_top": float(rel.max()),
            "cos": dict(zip(ids, map(float, cos))), "rel": dict(zip(pool, map(float, rel)))}

t0 = time.perf_counter(); out = [run(q["question"]) for q in qs]
out_un = [run(q) for q in UNANSWERABLE]
print(f"{(time.perf_counter() - t0) / (len(qs) + len(UNANSWERABLE)) * 1000:.0f} ms per question for all four rankers together (the cross-encoder scores 10 documents; the parts were not timed separately)")

5568 ms per question for all four rankers together (the cross-encoder scores 10 documents; the parts were not timed separately)


## Which ranker puts the right document first

Each ranker orders all 31 documents. The cross-encoder reranks the ten best of the fused list. The table counts a question as right at top-1 when the first document is a gold document, and reports the share whose top three contain a superseded or agent-only document.

In [3]:
SYS = ["bm25", "dense", "fused", "rerank"]
gold = [set(q["gold_doc_ids"]) for q in qs]
def hit(o, g, s, k): return int(bool(set(o[s][:k]) & g))
def mrr(o, g, s): return next((1 / (r + 1) for r, d in enumerate(o[s]) if d in g), 0.0)
res = {}
print(f"{'ranker':8s}{'top-1 right':>20s}{'gold in top 3':>22s}{'MRR':>7s}{'questions with a superseded or agent-only document in the top 3':>66s}")
for s in SYS:
    h1 = sum(hit(o, g, s, 1) for o, g in zip(out, gold)); h3 = sum(hit(o, g, s, 3) for o, g in zip(out, gold))
    bd = sum(bool(set(o[s][:3]) & bad) for o in out); mr = float(np.mean([mrr(o, g, s) for o, g in zip(out, gold)]))
    l1, u1 = m.wilson(h1, len(qs)); l3, u3 = m.wilson(h3, len(qs))
    res[s] = {"top1": h1, "top3": h3, "mrr": mr, "bad_top3": bd}
    print(f"{s:8s}{f'{h1}/{len(qs)} [{l1:.2f}, {u1:.2f}]':>20s}{f'{h3}/{len(qs)} [{l3:.2f}, {u3:.2f}]':>22s}{mr:>7.2f}{bd:>66d}")

ranker           top-1 right         gold in top 3    MRR   questions with a superseded or agent-only document in the top 3
bm25      13/20 [0.43, 0.82]    19/20 [0.76, 0.99]   0.80                                                                 8
dense     17/20 [0.64, 0.95]    20/20 [0.84, 1.00]   0.93                                                                11
fused     15/20 [0.53, 0.89]    20/20 [0.84, 1.00]   0.88                                                                10
rerank    18/20 [0.70, 0.97]    20/20 [0.84, 1.00]   0.95                                                                 7


Embedding similarity put a gold document first on 17 of 20 questions, the cross-encoder on 18, BM25 on 13 and the fusion on 15. The intervals overlap almost entirely, so these four numbers do not rank the methods; the gold document sits in the top three for 20 of 20 questions under similarity, fusion and reranking, and 19 of 20 under BM25. Fusion scored below similarity alone here. With 31 short documents both lists are already short, and BM25's seven top-1 misses can pull a correct similarity answer down; we did not test that explanation. A superseded or agent-only document reached the top three for 11 of 20 questions under similarity and 7 under reranking, so a filter on document metadata is needed whichever ranker is used.

## Where similarity and relevance disagree

In [4]:
rows = []
for q, g, o in zip(qs, gold, out):
    d1, r1 = o["dense"][0], o["rerank"][0]
    if d1 != r1: rows.append((q["query_type"], q["question"], d1 in g, r1 in g, d1, r1))
print(f"top-1 differs between embedding similarity and the cross-encoder on {len(rows)} of {len(qs)} questions\n")
for t, q, dok, rok, d1, r1 in rows:
    print(f"[{t}] {q}\n   similarity chose {d1:28s} {'right' if dok else 'WRONG'}\n   cross-encoder chose {r1:25s} {'right' if rok else 'WRONG'}")
b = sum(1 for r in rows if r[3] and not r[2]); c = sum(1 for r in rows if r[2] and not r[3])
from scipy.stats import binomtest
p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
print(f"\nsimilarity wrong and cross-encoder right: {b}; the reverse: {c}; exact sign test p = {p:.3f}")
res["disagree"] = {"n_differ": len(rows), "rerank_only_right": b, "dense_only_right": c, "p": p}

top-1 differs between embedding similarity and the cross-encoder on 4 of 20 questions

[stale_trap] Do I need any jabs for Bali?
   similarity chose health_bali_2023_archived    WRONG
   cross-encoder chose health_current            right
[stale_trap] How much is a checked bag?
   similarity chose baggage_2019_outdated        WRONG
   cross-encoder chose baggage_current           right
[multi_facet] What is there to do in Reykjavik?
   similarity chose dest_reykjavik               right
   cross-encoder chose excursion_suppliers       WRONG
[multi_facet] What excursions do you offer in the Norwegian fjords?
   similarity chose excursion_suppliers          right
   cross-encoder chose dest_oslo                 right

similarity wrong and cross-encoder right: 2; the reverse: 1; exact sign test p = 1.000


The two rankers choose a different first document on 4 of 20 questions. The cross-encoder is right where similarity is wrong on 2 (both stale traps), and wrong where similarity is right on 1 (it sent a question about things to do in Reykjavik to the excursion-supplier document). Three discordant questions cannot separate the two rankers; the sign test gives p = 1.0.

## The superseded document is the nearer one

A superseded document is written in the same words as the current one, so a similarity score has no reason to prefer the current one. The cell shows the top three under each ranker for the three stale-trap questions, and the two scores for the pair of baggage documents.

In [5]:
trap = [(q, o) for q, o in zip(qs, out) if q["query_type"] == "stale_trap"]
print(f"{len(trap)} stale-trap questions: a superseded document says something different from the current one.\n")
for q, o in trap:
    print(q["question"])
    for s in ("dense", "rerank"):
        top = o[s][:3]; print(f"   {s:7s} top 3: " + ", ".join(f"{d}{' (SUPERSEDED)' if d in bad else ''}" for d in top))
# the same documents, scored for the question that asks about the old fee
q = "How much is a checked bag?"
o = run(q)
print("\nSimilarity of that question to the two baggage documents, and the cross-encoder's relevance:")
for d in ("baggage_current", "baggage_2019_outdated"):
    print(f"   {d:24s} cosine {o['cos'][d]:.3f}   relevance {o['rel'].get(d, float('nan')):.4f}")

3 stale-trap questions: a superseded document says something different from the current one.

Do I need any jabs for Bali?
   dense   top 3: health_bali_2023_archived (SUPERSEDED), health_current, visa_bali
   rerank  top 3: health_current, visa_bali, dest_bali
How much is a checked bag?
   dense   top 3: baggage_2019_outdated (SUPERSEDED), baggage_current, baggage_sports
   rerank  top 3: baggage_current, baggage_2019_outdated (SUPERSEDED), baggage_sports
Do I need a COVID vaccination to travel to Iceland now?
   dense   top 3: health_covid_archived (SUPERSEDED), health_current, health_bali_2023_archived (SUPERSEDED)
   rerank  top 3: health_covid_archived (SUPERSEDED), health_current, excursion_suppliers



Similarity of that question to the two baggage documents, and the cross-encoder's relevance:
   baggage_current          cosine 0.734   relevance 0.9786
   baggage_2019_outdated    cosine 0.789   relevance 0.9236


Similarity put a superseded document first on all three stale traps. For "How much is a checked bag?" the 2019 fee schedule is the nearer document (cosine 0.789 against 0.734) because it states a fee in the question's own words. The cross-encoder put the current document first on two of the three, but it scored the pair 0.9786 and 0.9236: both documents address the question, and relevance does not include "still true". For the COVID question it ranked the archived 2021 rule first. Currency has to come from metadata (a current flag or an effective date) applied before ranking, as the RAG chapter's access and freshness filters do.

## Can a score gate "no answer"?

A refusal gate needs a score that is high when a document answers the question and low when none does, and a threshold set before the run. The cell takes the best document's score for the 20 answerable and the 6 unanswerable questions and reports AUROC (the chance that a random answerable question outscores a random unanswerable one; 0.5 is a coin flip), with a bootstrap interval, and the margin between the lowest answerable score and the highest unanswerable one.

In [6]:
def auroc(neg, pos, B=2000, seed=0):
    y = np.r_[np.zeros(len(neg)), np.ones(len(pos))]; a = roc_auc_score(y, np.r_[neg, pos])
    rng = np.random.default_rng(seed); bs = []
    for _ in range(B):
        i = rng.integers(0, len(neg), len(neg)); j = rng.integers(0, len(pos), len(pos))
        bs.append(roc_auc_score(np.r_[np.zeros(len(i)), np.ones(len(j))], np.r_[neg[i], pos[j]]))
    return a, float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

ans_cos = np.array([o["cos_top"] for o in out]); un_cos = np.array([o["cos_top"] for o in out_un])
ans_rel = np.array([o["rel_top"] for o in out]); un_rel = np.array([o["rel_top"] for o in out_un])
print(f"{'score of the best document':34s}{'answerable: min / median':>28s}{'unanswerable: median / max':>30s}{'AUROC [95% bootstrap]':>26s}")
for name, a, u in (("embedding similarity (cosine)", ans_cos, un_cos), ("cross-encoder relevance", ans_rel, un_rel)):
    au = auroc(u, a)    # does a high score separate answerable (positive) from unanswerable?
    res["gate_" + name.split()[0]] = {"auroc": au[0], "ci": [au[1], au[2]], "ans_min": float(a.min()), "un_max": float(u.max())}
    print(f"{name:34s}{f'{a.min():.3f} / {np.median(a):.3f}':>28s}{f'{np.median(u):.3f} / {u.max():.3f}':>30s}{f'{au[0]:.2f} [{au[1]:.2f}, {au[2]:.2f}]':>26s}")
print("\nunanswerable questions and the document each ranker liked best:")
for q, o in zip(UNANSWERABLE, out_un): print(f"   {q:60s} similarity {o['dense'][0]:22s} cross-encoder {o['rerank'][0]}")

score of the best document            answerable: min / median    unanswerable: median / max     AUROC [95% bootstrap]


embedding similarity (cosine)                    0.710 / 0.788                 0.658 / 0.696         1.00 [1.00, 1.00]


cross-encoder relevance                          0.056 / 0.966                 0.001 / 0.201         0.99 [0.95, 1.00]

unanswerable questions and the document each ranker liked best:
   How do I reset the password on my Meridian account?          similarity fare_conditions        cross-encoder refund_cancellation
   What time does the Oslo office open on Saturdays?            similarity dest_oslo              cross-encoder dest_oslo
   Who won the football match last night?                       similarity dest_tromso            cross-encoder dest_bergen
   What is the weather forecast for Oslo next week?             similarity dest_oslo              cross-encoder dest_oslo
   Is the northern lights tour in Tromso wheelchair accessible? similarity dest_tromso            cross-encoder dest_tromso
   How much does a hotel room in Paris cost tonight?            similarity baggage_2019_outdated  cross-encoder baggage_current


Neither score gives a gate with a margin. Cosine similarity separates the two groups perfectly (AUROC 1.00), but the lowest answerable score is 0.710 and the highest unanswerable one 0.696, a gap of 0.014 on 26 questions; a threshold placed there would move with the next question, and the bootstrap interval [1.00, 1.00] only says that these 26 questions do not overlap. The cross-encoder's AUROC is 0.99 (interval 0.95 to 1.00), and its ranges overlap: the lowest answerable score is 0.056 and the highest unanswerable one 0.201. We expected the cross-encoder to separate better, and on this set it did not. For the three same-topic unanswerable questions both rankers chose the Oslo or Tromso destination page, which is the topic match that neither score measures. Settling the question needs a few hundred labeled questions with same-topic unanswerable ones, a threshold fixed on one split and tested on another.

In [7]:
res["n_answerable"] = len(qs); res["n_unanswerable"] = len(UNANSWERABLE); res["ms_per_question_all_rankers"] = (time.perf_counter() - t0) / (len(qs) + len(UNANSWERABLE)) * 1000
Path("data/results/15_similarity_vs_relevance.json").write_text(json.dumps(res, indent=2, default=float))

920

## What to take from this

1. Similarity retrieves: the gold document was in its top three on 20 of 20 questions. Use it to build the candidate list, and add BM25 where users type exact strings such as booking codes (not tested here: no question in this set contains one).
2. Reranking changed the first document on 4 of 20 questions for a net gain of one; on 20 questions that cannot be told from chance. Its cost is one model pass per candidate, about 5 s per question for all four rankers together on a loaded laptop.
3. Filter on metadata first. Both scores treat a superseded document as a good answer because it is one, in the words the question uses.
4. A refusal threshold needs its own labeled split. The best-document score looked separable on 26 questions and had a margin of 0.014.